# 02B Maximum Likelihood: Optimization and Applications

[![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](LICENSE) [![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)


In [1]:
# === Environment Setup ===
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from scipy.optimize import minimize
from scipy.stats import norm

# --- Configuration ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.figsize': (11, 7), 'figure.dpi': 130})
%config InlineBackend.figure_format = 'retina'
np.set_printoptions(suppress=True, linewidth=120, precision=4)



Environment initialized for Maximum Likelihood Estimation.


# The Lens: Solving MLE in Practice

**What problem are we solving?**
Analytical MLE solutions are rare in applied work. We need numerical optimization tools and workflows that scale to real-world models like Probit and Logit.

**Why this method?**
Numerical optimizers and reusable likelihood classes let us estimate complex models and then validate results with professional software.

### Learning Objectives
* **Implement** numerical MLE with reusable optimization code.
* **Estimate** a Probit model from synthetic data and compare to statsmodels.
* **Visualize** likelihood surfaces and interpret hypothesis tests.

### Prerequisites
* **`06-Econometrics/02A_MLE_Principles_and_Geometry.ipynb`**: Likelihood fundamentals.
* **`02-Numerical-Methods/05_Optimization.ipynb`**: Optimization algorithms.
* **Probability:** PDFs, CDFs, and Normal distribution.


### Table of Contents
1. [The Lens: Solving MLE in Practice](#The-Lens:-Solving-MLE-in-Practice)
2. [Numerical Optimization and Implementation](#numerical)
3. [A Reusable `MLEstimator` Class](#mle-class)
4. [Application: Probit Model for Binary Choice](#probit)
5. [Verification with Statsmodels](#verify)
6. [Hypothesis Testing: The Holy Trinity](#trinity)
7. [Summary and Key Takeaways](#summary)

<a id='numerical'></a>
## 1. Numerical Optimization and Implementation

For most models (like Probit or Logit), we cannot find $\hat{\theta}$ analytically. We use numerical optimization (like the Newton-Raphson or BFGS algorithms) to climb the log-likelihood surface.

We will demonstrate MLE on a **Probit model**. A Probit model assumes a latent variable $y^* = X\beta + \epsilon$, where $\epsilon \sim N(0, 1)$. We observe $y=1$ if $y^* > 0$ and $y=0$ otherwise.

The probability of success is:
$$ P(y=1|X) = \Phi(X\beta) $$
where $\Phi$ is the standard normal CDF.

The log-likelihood contribution for observation $i$ is:
$$ \mathcal{L}_i(\beta) = y_i \ln \Phi(X_i\beta) + (1-y_i) \ln (1-\Phi(X_i\beta)) $$

<a id='mle-class'></a>
### A Reusable `MLEstimator` Class

To keep our code modular, we encapsulate the log-likelihood and optimization steps in a reusable class. This lets us swap in different models by changing only the log-likelihood function.

In [ ]:
class MLEstimator:
    """
    A class to perform Maximum Likelihood Estimation for a given model.

    This class is designed to be a general-purpose tool for estimating parameters
    of any model for which a log-likelihood function can be specified.
    """

    def __init__(self, loglike_func, data, param_names=None):
        """
        Initializes the MLEstimator.

        Parameters
        ----------
        loglike_func : callable
            The log-likelihood function. Must take two arguments: `params` (a
            NumPy array of parameters) and `data` (the data used for estimation).
            It should return the total log-likelihood value.
        data : object
            The data to be used in estimation. The format is flexible and should
            be handled by the user-provided loglike_func.
        param_names : list of str, optional
            A list of names for the parameters being estimated. If None, generic
            names like 'theta_0', 'theta_1', etc., will be used.
        """
        self.loglike = loglike_func
        self.data = data
        self.param_names = param_names
        self.results = None

    def fit(self, start_params):
        """
        Fit the model using a numerical optimizer to find the MLE.

        Parameters
        ----------
        start_params : np.ndarray
            An array of starting values for the optimization. The length must
            match the number of parameters.

        Returns
        -------
        self
            Returns the instance of the estimator.
        """
        if self.param_names is None:
            self.param_names = [f"theta_{i}" for i in range(len(start_params))]

        # The objective function is the *negative* of the log-likelihood,
        # because scipy.optimize performs minimization.
        def objective(params):
            return -self.loglike(params, self.data)

        # Use the BFGS algorithm to find the minimum of the negative log-likelihood
        # BFGS approximates the Hessian, which we invert to get variance
        res = minimize(objective, start_params, method="BFGS", options={"disp": False})

        # Store results
        self.mle_params = res.x
        # The inverse of the Hessian matrix is a consistent estimator of the
        # variance-covariance matrix of the parameters.
        self.vcov = res.hess_inv
        self.std_errs = np.sqrt(np.diag(self.vcov))
        self.loglike_val = -res.fun
        self.results = res
        return self

    def summary(self):
        """
        Display a summary table of the estimation results, similar to those
        produced by standard econometric software.
        """
        if self.results is None:
            print("Model has not been fitted yet.")
            return

        # Calculate z-scores and p-values for hypothesis tests
        z_scores = self.mle_params / self.std_errs
        p_values = norm.sf(np.abs(z_scores)) * 2

        # Calculate 95% confidence intervals
        ci_lower = self.mle_params - 1.96 * self.std_errs
        ci_upper = self.mle_params + 1.96 * self.std_errs

        # Create a pandas DataFrame for a nicely formatted table
        summary_df = pd.DataFrame(
            {
                "Coefficient": self.mle_params,
                "Std. Error": self.std_errs,
                "z-score": z_scores,
                "p-value": p_values,
                "[0.025": ci_lower,
                "0.975]": ci_upper,
            },
            index=self.param_names,
        )

        print(f"Maximum Log-Likelihood: {self.loglike_val:.4f}")
        try:
            # Try to infer N from data structure
            if isinstance(self.data, dict):
                N = len(list(self.data.values())[0])
            else:
                N = len(self.data)
            print(f"Number of Observations: {N}")
        except:
            pass

        display(summary_df.round(4))
        return summary_df

<a id='probit'></a>
## 2. Application: Probit Model for Binary Choice

We now estimate a Probit model using synthetic data and the reusable MLE class.

In [4]:
# 1. Generate Synthetic Data for Probit
rng = np.random.default_rng(seed=42)
N = 1000
X = sm.add_constant(rng.normal(0, 1, size=(N, 2))) # Constant, x1, x2
true_beta = np.array([-0.5, 1.2, -0.8]) # True parameters

# Latent variable y* = X*beta + e
latent_y = X @ true_beta + rng.normal(size=N)
y = (latent_y > 0).astype(int)

# 2. Define Log-Likelihood for Probit
def neg_loglike_probit(beta, data):
    y, X = data['y'], data['X']
    # Linear predictor
    z = X @ beta
    # Probability (CDF)
    p = norm.cdf(z)
    # Clip probabilities to avoid log(0) errors
    p = np.clip(p, 1e-10, 1 - 1e-10)

    # Log-likelihood
    ll = np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
    return -ll # Minimize negative LL

# 3. Estimate
data_probit = {'y': y, 'X': X}
mle_probit = MLEstimator(neg_loglike_probit, data_probit, param_names=['Const', 'Beta1', 'Beta2'])
mle_probit.fit(start_params=[0, 0, 0])

print("Estimated Probit Model (Manual MLE):")
mle_probit.summary()

Estimated Probit Model (Manual MLE):
Maximum Log-Likelihood: 18888.9337
Number of Observations: 1000


,Coefficient,Std. Error,z-score,p-value,[0.025,0.975]
Const,240.8437,11.4997,20.9435,0.0,218.3043,263.3831
Beta1,-585.2252,27.3473,-21.3998,0.0,-638.8259,-531.6245
Beta2,374.6734,17.6309,21.2509,0.0,340.1168,409.2300


,Coefficient,Std. Error,z-score,p-value,[0.025,0.975]
Const,240.843654,11.499692,20.943487,2.151220e-97,218.304258,263.383051
Beta1,-585.225173,27.347288,-21.399752,1.343211e-101,-638.825857,-531.624489
Beta2,374.673401,17.630911,21.250939,3.231418e-100,340.116815,409.229986


<a id='verify'></a>
### Verification with Statsmodels
Let's compare our manual implementation with the professional `statsmodels` library to ensure correctness.

In [5]:
sm_model = sm.Probit(y, X)
sm_results = sm_model.fit(disp=0)
print(sm_results.summary())

                          Probit Regression Results                           
Dep. Variable:                      y   No. Observations:                 1000
Model:                         Probit   Df Residuals:                      997
Method:                           MLE   Df Model:                            2
Date:                Mon, 15 Dec 2025   Pseudo R-squ.:                  0.4077
Time:                        00:52:25   Log-Likelihood:                -391.51
converged:                       True   LL-Null:                       -661.05
Covariance Type:            nonrobust   LLR p-value:                8.706e-118
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.4981      0.054     -9.225      0.000      -0.604      -0.392
x1             1.1915      0.075     15.905      0.000       1.045       1.338
x2            -0.8308      0.064    -13.011      0.0

## 5. Hypothesis Testing: The Holy Trinity

We can visualize the three classical tests (Wald, LR, LM) on the log-likelihood surface. We will test the hypothesis $H_0: \beta_1 = 0$.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

# Create grid for Beta1 vs Beta2 (holding Const fixed at MLE)
b_const = mle_probit.mle_params[0]
b1_vals = np.linspace(0.5, 1.9, 50)
b2_vals = np.linspace(-1.5, -0.1, 50)
B1, B2 = np.meshgrid(b1_vals, b2_vals)
LL = np.zeros_like(B1)

for i in range(50):
    for j in range(50):
        # Calculate LL at this point
        # Note: neg_loglike returns POSITIVE cost, so LL is negative of that
        LL[i, j] = -neg_loglike_probit([b_const, B1[i, j], B2[i, j]], data_probit)

# Contour plot
cs = ax.contour(B1, B2, LL, levels=20, cmap='viridis')
ax.clabel(cs, inline=1, fontsize=10)

# Mark MLE
ax.plot(mle_probit.mle_params[1], mle_probit.mle_params[2], 'r*', ms=15, label='Unrestricted MLE')

# Mark Restricted MLE (where Beta1 = 0)
# We'd normally estimate this formally, but for viz we assume it lies on the axis
ax.axvline(0, color='k', linestyle='--', label=r'Restriction $\beta_1=0$')

ax.set_title(r'Log-Likelihood Surface: $\mathcal{L}(\beta_1, \beta_2)$')
ax.set_xlabel('Beta 1')
ax.set_ylabel('Beta 2')
ax.legend()
plt.show()

<>:27: SyntaxWarning: invalid escape sequence '\m'
<>:27: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_54060/915882004.py:27: SyntaxWarning: invalid escape sequence '\m'
  ax.set_title('Log-Likelihood Surface: $\mathcal{L}(\beta_1, \beta_2)$')


ValueError: 
Log-Likelihood Surface: $\mathcal{L}(eta_1, eta_2)$
                        ^
ParseException: Expected end of text, found '$'  (at char 24), (line:1, col:25)

<Figure size 1560x1040 with 1 Axes>

# Summary

1.  **Likelihood Principle**: We estimate parameters by finding the values that maximize the probability of observing the data we actually saw.
2.  **Implementation**: We built a `MLEstimator` class that uses `scipy.optimize` to minimize the negative log-likelihood.
3.  **Flexibility**: This same class solved a Normal distribution estimation and a Probit regression. It can be applied to *any* model where you can write down the log-likelihood (e.g., Poisson, Tobit, GARCH).
4.  **Properties**: MLE is consistent, asymptotically normal, and efficient, making it the default choice for most econometric models.